# Transformers, what can they do?

This notebook explores the **Hugging Face `transformers` library** — a powerful toolkit that lets you use
pre-trained transformer models for a wide range of NLP (Natural Language Processing) tasks with just a few lines of code.

### Tasks covered in this notebook:
1. **Sentiment Analysis** — Is a text positive or negative?
2. **Zero-Shot Classification** — Classify text into custom categories without any training
3. **Text Generation** — Continue a piece of text automatically
4. **Fill-Mask** — Predict missing words in a sentence
5. **Named Entity Recognition (NER)** — Detect names, places, organizations in text
6. **Question Answering** — Extract answers from a passage of text
7. **Summarization** — Shorten a long text while keeping key information
8. **Translation** — Translate text between languages

## Setup

Install the required libraries before running any cells.
- `transformers` — the main HuggingFace library with all pre-trained models and pipelines
- `datasets` — tools for loading and processing text datasets
- `evaluate` — metrics for evaluating model performance
- `sentencepiece` — tokenizer used by many multilingual models

In [1]:
# Install all required libraries
# Run this cell once before anything else
# The [sentencepiece] extra installs the SentencePiece tokenizer,
# needed for multilingual and translation models
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.5 MB/s eta 0:00:00


## 1. Sentiment Analysis

Sentiment analysis classifies text as **POSITIVE** or **NEGATIVE** (sometimes NEUTRAL).

**How it works:** The model reads the entire sentence and assigns a label + confidence score (0.0–1.0).
A score of 0.96 means the model is 96% confident in its prediction.

**Common use cases:** Product review analysis, customer feedback triage, social media monitoring.

In [2]:
from transformers import pipeline

# pipeline() is the simplest way to use a pre-trained model.
# 'sentiment-analysis' automatically downloads and loads a fine-tuned
# BERT model trained on the SST-2 dataset (Stanford Sentiment Treebank).
classifier = pipeline("sentiment-analysis")

# Pass a single sentence — returns a list with one dict:
# 'label': 'POSITIVE' or 'NEGATIVE'
# 'score': confidence between 0.0 and 1.0
classifier("I've been waiting for a HuggingFace course my whole life.")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9598048329353333}]

In [3]:
# You can also pass a LIST of sentences for batch processing.
# The model processes all inputs together — much faster than calling it
# one sentence at a time in a loop.
# Output: a list of dicts, one per input sentence.
classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",  # → POSITIVE
        "I hate this so much!"  # → NEGATIVE
    ]
)

[{'label': 'POSITIVE', 'score': 0.9598048329353333},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

## 2. Zero-Shot Classification

Classify text into **custom categories you define at runtime** — no model training required!

**How it works:** The model (usually MNLI-trained) checks how well the text *entails* each label.
You provide the candidate labels; the model scores each one.

**Why it's powerful:** Normally, a classifier only works for the categories it was trained on.
Zero-shot classification lets you define ANY labels and get a reasonable prediction immediately.

**Common use cases:** Content moderation, topic tagging, intent detection with custom categories.

In [4]:
from transformers import pipeline

# 'zero-shot-classification' loads a model trained on Natural Language Inference (NLI).
# It reframes classification as: "Does this text imply it belongs to label X?"
classifier = pipeline("zero-shot-classification")

classifier(
    "This is a course about the Transformers library",

    # candidate_labels: define your own categories — these are not fixed!
    # The model will score each label and rank them by probability.
    # Output scores always sum to 1.0 across all labels.
    candidate_labels=["education", "politics", "business"],
)
# Expected output:
# education: 84.5% — most likely
# business:  11.2%
# politics:   4.3% — least likely

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.8445985913276672, 0.11197450757026672, 0.04342687129974365]}

## 3. Text Generation

Given a text prompt, the model **continues writing** by predicting the most likely next tokens.

**How it works:** A GPT-style (decoder-only) language model generates tokens one at a time,
each token conditioned on all previous ones.

**Common use cases:** Autocomplete, story writing, code generation, content drafting.

In [5]:
from transformers import pipeline

# 'text-generation' loads GPT-2 by default.
# The model will extend the given prompt with new text.
# Note: output is non-deterministic — you'll get different results each run!
generator = pipeline("text-generation")

# The prompt acts as the starting context.
# The model generates tokens until it reaches max_length or a stop token.
generator("In this course, we will teach you how to")

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'In this course, we will teach you how to use the Power Tool, to quickly and easily identify and install a Power Tool on your system.'}]

In [6]:
from transformers import pipeline

# Using a specific smaller model: 'distilgpt2'
# distilgpt2 is a distilled (compressed) version of GPT-2 — faster and lighter,
# but slightly less capable than full GPT-2.
generator = pipeline("text-generation", model="distilgpt2")

generator(
    "In this course, we will teach you how to",

    # max_length: stop generating after this many total tokens (prompt + generated)
    # Lower = faster, shorter output. Higher = longer output but more compute.
    max_length=30,

    # num_return_sequences: generate N different completions for the same prompt.
    # Useful for exploring multiple creative continuations.
    num_return_sequences=2,
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'In this course, we will teach you how to use an old school method of arithmetic.'},
 {'generated_text': 'In this course, we will teach you how to use one of the most powerful tools in Linux. You will then use the following tools to help you build and upgrade your project.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'}]

## 4. Fill-Mask (Masked Language Modeling)

Predict what word should replace a `<mask>` token in a sentence.

**How it works:** BERT-style (encoder) models are trained by randomly masking tokens and
learning to predict them. At inference time, you deliberately place `<mask>` where you
want a prediction.

**Common use cases:** Auto-correct, vocabulary suggestion, probing what a model has learned,
data augmentation.

In [7]:
from transformers import pipeline

# 'fill-mask' loads a BERT-based model trained with masked language modeling (MLM).
# The special token <mask> marks the position where a word should be predicted.
unmasker = pipeline("fill-mask")

unmasker(
    # Place <mask> exactly where you want the model to predict a word.
    # The model uses BOTH left and right context (bidirectional attention)
    # to make its prediction — unlike GPT which only sees left context.
    "This course will teach you all about <mask> models.",

    # top_k: return the top K most likely predictions with their scores.
    # Higher k = more candidate words to compare.
    top_k=2,
)
# 'mathematical' scores highest because BERT has seen many
# "X models" patterns in its training data.

No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[{'score': 0.19619713723659515,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.040527280420064926,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

## 5. Named Entity Recognition (NER)

Identify and classify **named entities** in text — people, organizations, locations, dates, etc.

**How it works:** A token-classification model assigns an entity label to each word (or subword).
Common labels:
- `PER` — Person name
- `ORG` — Organization
- `LOC` — Location
- `MISC` — Miscellaneous (events, nationalities, etc.)

**Common use cases:** Information extraction, knowledge graph construction, document parsing.

In [8]:
from transformers import pipeline

# 'ner' = Named Entity Recognition
# aggregation_strategy="simple" merges consecutive tokens that belong to the same entity.
# Without aggregation, multi-word entities like 'Hugging Face' would appear as
# two separate tokens: 'Hugging' and 'Face'.
ner = pipeline("ner", aggregation_strategy="simple")

ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

# Each result dict contains:
# 'entity_group': the entity type (PER, ORG, LOC, MISC)
# 'score':        model confidence (0.0–1.0)
# 'word':         the detected entity text
# 'start'/'end':  character positions in the original string

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'entity_group': 'PER',
  'score': np.float32(0.9981694),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': np.float32(0.9796019),
  'word': 'Hugging Face',
  'start': 33,
  'end': 45},
 {'entity_group': 'LOC',
  'score': np.float32(0.9932106),
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

## 6. Question Answering (Extractive)

Given a question and a context passage, **extract the exact answer span** from the context.

**How it works:** The model predicts two positions in the context: the start and end of the answer.
This is called *extractive* QA — the answer must exist verbatim in the context.
(Contrast with *generative* QA like ChatGPT, which can generate new text as an answer.)

**Common use cases:** Document search, FAQ automation, reading comprehension systems.

In [9]:
from transformers import pipeline

# 'question-answering' loads a model fine-tuned on SQuAD
# (Stanford Question Answering Dataset) — a benchmark for extractive QA.
question_answerer = pipeline("question-answering")

question_answerer(
    # The question you want answered
    question="Where do I work?",

    # The context passage that contains the answer.
    # The model will NOT answer from general knowledge —
    # it can only extract text that exists in this context string.
    context="My name is Sylvain and I work at Hugging Face in Brooklyn",
)
# Returns:
# 'answer': the extracted text ('Hugging Face')
# 'score':  confidence that this span is the correct answer
# 'start'/'end': character index of the answer in context

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'score': 0.6949770450592041, 'start': 33, 'end': 45, 'answer': 'Hugging Face'}

## 7. Summarization

Condense a long text into a shorter version while preserving key information.

**How it works:** Uses a sequence-to-sequence (encoder-decoder) model like BART or T5.
The encoder reads the full text; the decoder generates a shorter summary token by token.

**Common use cases:** News digest, document briefing, meeting notes, research paper abstracts.

## 8. Translation

Translate text from one language to another using a pre-trained neural machine translation model.

**How it works:** Sequence-to-sequence encoder-decoder architecture.
The Helsinki-NLP models (Opus-MT) are trained on the OPUS parallel corpus — millions of
aligned sentence pairs across 1,000+ language pairs.

**Common use cases:** Document translation, multilingual chatbots, cross-language search.

## Summary

| Task | Pipeline name | Default model | Output |
|------|--------------|---------------|--------|
| Sentiment Analysis | `sentiment-analysis` | distilbert-sst2 | label + score |
| Zero-Shot Classification | `zero-shot-classification` | bart-large-mnli | labels + scores |
| Text Generation | `text-generation` | gpt2 | generated text |
| Fill-Mask | `fill-mask` | distilroberta | top-k completions |
| Named Entity Recognition | `ner` | bert-base-NER | entity spans |
| Question Answering | `question-answering` | distilbert-squad | answer span |
| Summarization | `summarization` | bart-large-cnn | summary text |
| Translation | `translation` | opus-mt-xx-yy | translated text |

All of these use the same simple API: `pipeline(task)` → call with text → get results.
The real power comes when you swap in specialized models for your domain!